# Financial Advisor Exploration

Run these cells from top to bottom. This notebook independently explores validation, Yahoo Finance data, and educational report generation. It does not place trades.

In [1]:
import yfinance as yf

print('yfinance imported')

yfinance imported


In [2]:
from dataclasses import dataclass
from datetime import datetime, timezone

print('standard library imports ready')

standard library imports ready


## 1. Define allowed profile values

In [3]:
RISK_LEVELS = {'very conservative', 'conservative', 'balanced', 'aggressive', 'very aggressive'}
HORIZONS = {'intraday', 'short-term', 'medium-term', 'long-term'}
DISCLAIMER = 'Educational information only. This is not financial advice.'
print(sorted(RISK_LEVELS))
print(sorted(HORIZONS))

['aggressive', 'balanced', 'conservative', 'very aggressive', 'very conservative']
['intraday', 'long-term', 'medium-term', 'short-term']


## 2. Validate an advisor request

In [4]:
def validate_request(ticker: str, risk: str, period: str) -> tuple[str, str, str]:
    """Normalize and validate ticker, risk attitude, and horizon."""
    clean_ticker = ticker.strip().upper()
    if not clean_ticker or len(clean_ticker) > 10:
        raise ValueError('Ticker must contain 1-10 characters.')
    clean_risk = risk.strip().lower()
    if clean_risk not in RISK_LEVELS:
        raise ValueError('Unsupported risk attitude.')
    clean_period = period.strip().lower()
    if clean_period not in HORIZONS:
        raise ValueError('Unsupported investment period.')
    return clean_ticker, clean_risk, clean_period

In [5]:
request = validate_request(' aapl ', 'Conservative', 'Short-term')
print(request)
try:
    validate_request('AAPL', 'unknown', 'long-term')
except ValueError as error:
    print(f'Validation passed: {error}')

('AAPL', 'conservative', 'short-term')
Validation passed: Unsupported risk attitude.


## 3. Fetch a read-only Yahoo Finance snapshot

In [6]:
@dataclass(frozen=True)
class MarketSnapshot:
    ticker: str
    latest_price: float
    daily_change_percent: float | None
    period_high: float
    period_low: float
    latest_volume: int
    as_of: str

def fetch_snapshot(ticker: str) -> MarketSnapshot:
    """Fetch one month of daily history without placing trades."""
    history = yf.Ticker(ticker).history(period='1mo', interval='1d', auto_adjust=False)
    closes = history['Close'].dropna()
    volumes = history['Volume'].dropna()
    if closes.empty:
        raise RuntimeError('No market data returned.')
    latest = float(closes.iloc[-1])
    previous = float(closes.iloc[-2]) if len(closes) > 1 else None
    change = ((latest - previous) / previous * 100) if previous else None
    return MarketSnapshot(ticker, latest, change, float(closes.max()), float(closes.min()), int(volumes.iloc[-1]), datetime.now(timezone.utc).isoformat())

In [ ]:
try:
    snapshot = fetch_snapshot('AAPL')
    print(snapshot)
except Exception as error:
    print(f'Market data unavailable: {error}')

## 4. Create an educational report

In [7]:
def create_educational_report(ticker: str, risk: str, period: str, snapshot: MarketSnapshot | None) -> dict:
    """Combine market context with non-personalized planning guidance."""
    market = 'No snapshot available.' if snapshot is None else f'Latest close: ${snapshot.latest_price:.2f}; range: ${snapshot.period_low:.2f}-${snapshot.period_high:.2f}'
    return {
        'disclaimer': DISCLAIMER,
        'market_analysis': f'{ticker}: {market}',
        'strategy': f'Create a {period} observation plan aligned with a {risk} profile.',
        'safety': 'Use paper trading first. This notebook cannot place orders.'
    }

In [8]:
report = create_educational_report(request[0], request[1], request[2], snapshot if 'snapshot' in globals() else None)
print(report)

{'disclaimer': 'Educational information only. This is not financial advice.', 'market_analysis': 'AAPL: No snapshot available.', 'strategy': 'Create a short-term observation plan aligned with a conservative profile.', 'safety': 'Use paper trading first. This notebook cannot place orders.'}
